# 2.  Data Preparation

In [36]:
import sys
sys.path.insert(0, "..")
import os
import importlib
import src.code.io_utils as io
import src.code.data_load as dl
import src.code.data_preparation as dp

importlib.reload(io)
importlib.reload(dl)
importlib.reload(dp)

<module 'src.code.data_preparation' from 'c:\\Users\\User\\Desktop\\mestrado\\1st_year\\2nd_semester\\BC\\BC_Siemens\\Siemens_Advanta_BC\\notebooks\\..\\src\\code\\data_preparation.py'>

## 1. Load Datasets

In [37]:
training = dl.load_training()
validation = dl.load_validation()
macro_data = dl.load_macro_data()
period_data = dl.load_period_data()
bus=dl.load_bus_parsed()

## 2. Training & Validation Cleaning

### Drop Constant Columns

In [38]:
column_to_drop=['TGL Biz Desc']

In [39]:
training   = training.drop(columns=column_to_drop)
validation = validation.drop(columns=column_to_drop)

In [40]:
training   = dp.drop_high_missing_rows(training)


Dropped 0 rows with >50% missing values


In [41]:
validation = dp.drop_high_missing_rows(validation)

Dropped 0 rows with >50% missing values


### Categorical Validation Against BUs

In [42]:
dp.validate_categorical_codes(training,   bus, "Training")


CATEGORICAL VALIDATION — Training
                 Column  Unique in data  Valid in BUs  Invalid codes Invalid values
      TGL Business Unit               4             4              0    ✓ All valid
   TGL Business Segment              24            24              0    ✓ All valid
TGL Business Subsegment             134           134              0    ✓ All valid


,Column,Unique in data,Valid in BUs,Invalid codes,Invalid values
0,TGL Business Unit,4,4,0,✓ All valid
1,TGL Business Segment,24,24,0,✓ All valid
2,TGL Business Subsegment,134,134,0,✓ All valid


In [43]:
dp.validate_categorical_codes(validation, bus, "Validation")


CATEGORICAL VALIDATION — Validation
                 Column  Unique in data  Valid in BUs  Invalid codes Invalid values
      TGL Business Unit               4             4              0    ✓ All valid
   TGL Business Segment              20            20              0    ✓ All valid
TGL Business Subsegment             124           123              1   [SSI0278218]


,Column,Unique in data,Valid in BUs,Invalid codes,Invalid values
0,TGL Business Unit,4,4,0,✓ All valid
1,TGL Business Segment,20,20,0,✓ All valid
2,TGL Business Subsegment,124,123,1,[SSI0278218]


All categorical codes (`TGL Business Unit`, `TGL Business Segment`, `TGL Business Subsegment`) in the **training set** are consistent with the `BUs` reference hierarchy, no invalid codes found.

In the **validation set**, one subsegment was flagged as invalid: `SSI0278218`. This code belongs to a valid Segment (`SSI02782`) and BU (`SSI027`), but does not appear in the `BUs` reference file. It corresponds to 3 rows (periods 46–48) with missing `Orders` and `Revenue` — i.e., exactly the values to be predicted.

This is a **cold start** problem: the model has no historical data for this subsegment. Possible approaches include using the parent Segment forecast as a proxy or averaging sibling subsegments. This needs to be flagged to the course forum for clarification.

## 3. Macro Data Preparation

In [44]:
macro_data = dp.drop_high_missing_rows(macro_data)


Dropped 2 rows with >50% missing values


In [45]:
period_data = dp.drop_high_missing_rows(period_data)

Dropped 0 rows with >50% missing values


### GDP Imputation (Annual & Quarterly)

In [46]:
macro_data = dp.impute_gdp(macro_data, period_data)

# Verificar que os missings GDP foram tratados
gdp_cols = [c for c in macro_data.columns if 'GDP' in c]
print(macro_data[gdp_cols].isna().sum())

China_GDP                                3
China_GDP_from_Construction              0
China_GDP_from_Manufacturing             0
France_GDP                               3
France_GDP_from_Construction             0
France_GDP_from_Manufacturing            0
Germany_GDP                              3
Germany_GDP_from_Construction            0
Germany_GDP_from_Manufacturing           0
Italy_GDP                                3
Italy_GDP_from_Construction              0
Italy_GDP_from_Manufacturing             0
Japan_GDP                                3
Japan_GDP_from_Construction              3
Japan_GDP_from_Manufacturing             3
Switzerland_GDP                          3
United_Kingdom_GDP                       3
United_Kingdom_GDP_from_Construction     0
United_Kingdom_GDP_from_Manufacturing    0
United_States_GDP                        3
United_States_GDP_from_Construction      0
United_States_GDP_from_Manufacturing     0
dtype: int64


GDP values are not reported monthly but follow a fixed publication calendar:

- **Annual GDP** (e.g. `China_GDP`): reported in **December only** → backfilled within each calendar year (limit = 11 months)
- **Quarterly GDP** (e.g. `China_GDP_from_Manufacturing`): reported in **March, June, September, December** → backfilled within each quarter (limit = 2 months)
- **Exception**: `Japan_GDP_from_Construction` and `Japan_GDP_from_Manufacturing` follow an **annual** pattern (December only) despite being `GDP_from` columns

This is a deterministic business rule, not statistical imputation, so it is applied here rather than in the modeling pipeline.

The **3 remaining missing values** per annual GDP column correspond to **January, February, and March 2025** (periods 46–48), which fall beyond the last available December. These will be handled by the pipeline imputer.

### Sporadic Missing Value Imputation

In [47]:
macro_data = dp.impute_macro_sporadic(macro_data)

# Verificar que nao ha mais missings (excepto os 3 GDP de jan-mar 2025)
print(macro_data.isna().sum()[macro_data.isna().sum() > 0])

China_GDP                           3
China_Industrial_Production_Mom     8
China_Interest_Rate                40
France_GDP                          3
France_Interest_Rate                1
Germany_GDP                         3
Germany_Interest_Rate               1
Italy_GDP                           3
Italy_Interest_Rate                 1
Japan_GDP                           3
Japan_GDP_from_Construction         3
Japan_GDP_from_Manufacturing        3
Japan_Interest_Rate                 1
Switzerland_GDP                     3
Switzerland_Interest_Rate           1
United_Kingdom_GDP                  3
United_Kingdom_Interest_Rate        1
United_States_GDP                   3
United_States_Interest_Rate         1
dtype: int64


In [48]:
# check for missing values during the relevant periods
relevant_macro = macro_data[macro_data['Period'].between(43, 48)]
print(relevant_macro.isnull().sum()[relevant_macro.isnull().sum() > 0])

China_GDP                       3
France_GDP                      3
Germany_GDP                     3
Italy_GDP                       3
Japan_GDP                       3
Japan_GDP_from_Construction     3
Japan_GDP_from_Manufacturing    3
Switzerland_GDP                 3
United_Kingdom_GDP              3
United_States_GDP               3
dtype: int64


After GDP imputation, the remaining missing values (~1-2%) in monthly indicators 
(e.g. Inflation Rate, Exports, Interest Rate) were treated with **forward fill (ffill)**.

This approach propagates the last valid observed value to the next period, which is 
a standard and defensible strategy for macroeconomic time series where values tend 
to be stable between reporting dates.

Missing values at the beginning of the series (periods -131 to ~-91) were intentionally 
left untreated, as these periods fall outside the training and validation range (1–48) 
and will not be used in the merge.

The 3 remaining missing values per annual GDP column (periods 46–48, Jan–Mar 2025) 
are also left untreated and will be handled by the pipeline imputer, as no future 
GDP data is available to fill them deterministically.

## 4. Dataset Integration

### Merge Training & Validation with Macro Data

In [49]:
training_merged   = dp.merge_with_macro(training,   macro_data)
validation_merged = dp.merge_with_macro(validation, macro_data)

print(training.shape)    #(4237, 7 + 77 - 1 macro cols)
print(validation.shape)  #(715, 7 + 77 - 1 macro cols)

(4237, 6)
(715, 6)


## 5. Hierarchical Aggregation

### Create 4 Levels (Subsegment, Segment, BU, Total)

In [50]:
hierarchy_training   = dp.create_hierarchy(training_merged)
hierarchy_validation = dp.create_hierarchy(validation_merged)

# Verificar shapes
for level, df in hierarchy_training.items():
    print(f"{level:>12}: {df.shape}")


  subsegment: (4237, 83)
     segment: (791, 82)
          bu: (168, 81)
       total: (42, 80)


The merged training and validation datasets were aggregated into 4 hierarchical levels:

- **Subsegment** (4237 × 83): base level, no aggregation applied
- **Segment** (791 × 82): Orders and Revenue summed across Subsegments within each Segment and Period
- **BU** (168 × 81): Orders and Revenue summed across Segments within each BU and Period
- **Total** (42 × 80): Orders and Revenue summed across all BUs for each Period

Macro economic columns are identical within the same period and are therefore carried forward using `first()` at each aggregation level, they are not summed.

These 4 levels support all hierarchical forecasting approaches (Bottom-Up, Top-Down, Middle-Out, and Reconciliation), providing flexibility for the modeling phase.

## 6. Save Datasets

In [51]:
os.makedirs("../data/prepared", exist_ok=True)

for level, df in hierarchy_training.items():
    df.to_parquet(f"../data/prepared/training_{level}.parquet", index=False)

for level, df in hierarchy_validation.items():
    df.to_parquet(f"../data/prepared/validation_{level}.parquet", index=False)

# Save macro_data for feature engineering
macro_data.to_parquet("../data/prepared/macro_data_clean.parquet", index=False)